# Avocodos 2026 Project

In [1]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [2]:
# Create timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print ("Thang Tran")

Current Time:09_27_2026_1241AM
Thang Tran


In [3]:
# Read the 2025 and 2026 datasets
data2025 = pd.read_csv("2025data.csv", encoding="utf-8")
data2026 = pd.read_csv("2026data.csv", encoding="utf-8")

In [4]:
# Check the size of both datasets
print("2025:", data2025.shape)
print("2026:", data2026.shape)

2025: (6656, 16)
2026: (4096, 16)


In [5]:
# Check whether both datasets have the same columns
data2025.columns.equals(data2026.columns)

True

In [6]:
# Label each dataset by year
data2025["Year"] = 2025
data2026["Year"] = 2026

# Combine both datasets
data = pd.concat([data2025, data2026], ignore_index=True)

# Check the combined data
data["Year"].value_counts()

Year
2025    6656
2026    4096
Name: count, dtype: int64

In [7]:
# Keep only Total U.S. data
us_data = data[data["Geography"] == "Total U.S."]

# Keep weeks 1-32 so both years cover the same period
comparison_data = us_data[us_data["Period"] <= 32]

# Verify total U.S. avocado volume
total_volume = comparison_data.groupby("Year")["Total Bulk and Bags"].sum()

total_volume

Year
2025    1.870699e+09
2026    2.125800e+09
Name: Total Bulk and Bags, dtype: float64

In [8]:
# Check the range of periods available in each year
us_data.groupby("Year")["Period"].agg(["min", "max", "nunique"])


,min,max,nunique
Year,,,
2025,1,52,52
2026,1,32,32


In [9]:
# Verify overall volume and weighted average selling price changes
volume_change = (
    (total_volume[2026] - total_volume[2025])
    / total_volume[2025] * 100
)

weighted_price = comparison_data.groupby("Year").apply(
    lambda x: (x["ASP Current Year"] * x["Total Bulk and Bags"]).sum()
              / x["Total Bulk and Bags"].sum(),
    include_groups=False
)

price_change = (
    (weighted_price[2026] - weighted_price[2025])
    / weighted_price[2025] * 100
)

print("Volume change:", volume_change)
print("Weighted ASP:")
print(weighted_price)
print("Price change:", price_change)

Volume change: 13.636685015813757
Weighted ASP:
Year
2025    1.245151
2026    1.007930
dtype: float64
Price change: -19.051612585031556


In [17]:
# Verify 2026 weekly weighted average price range
weekly_2026 = comparison_data[comparison_data["Year"] == 2026].groupby("Period").apply(
    lambda x: (x["ASP Current Year"] * x["Total Bulk and Bags"]).sum()
              / x["Total Bulk and Bags"].sum(),
    include_groups=False
)

print(f"Minimum weekly weighted price: ${weekly_2026.min():.2f}")
print(f"Maximum weekly weighted price: ${weekly_2026.max():.2f}")

Minimum weekly weighted price: $0.92
Maximum weekly weighted price: $1.21


In [11]:
# Verify Conventional and Organic volume, price, and changes together
type_summary = comparison_data.groupby(["Year", "Type"]).apply(
    lambda x: pd.Series({
        "Volume": x["Total Bulk and Bags"].sum(),
        "Weighted ASP": (
            (x["ASP Current Year"] * x["Total Bulk and Bags"]).sum()
            / x["Total Bulk and Bags"].sum()
        )
    }),
    include_groups=False
)

type_change = (
    (type_summary.loc[2026] - type_summary.loc[2025])
    / type_summary.loc[2025] * 100
)

organic_share = (
    type_summary.loc[(slice(None), "Organic"), "Volume"]
    .droplevel("Type") / total_volume * 100
)

print("Summary:")
print(type_summary)

print("\nPercent Changes:")
print(type_change)

print("\nOrganic Share:")
print(organic_share)


Summary:
                         Volume  Weighted ASP
Year Type                                    
2025 Conventional  1.764839e+09      1.215428
     Organic       1.058595e+08      1.740676
2026 Conventional  2.020685e+09      0.979122
     Organic       1.051156e+08      1.561718

Percent Changes:
                 Volume  Weighted ASP
Type                                 
Conventional  14.496802    -19.442237
Organic       -0.702773    -10.280981

Organic Share:
Year
2025    5.658822
2026    4.944753
dtype: float64


In [16]:
# Verify change in total bagged avocado volume
bagged_volume = comparison_data.groupby("Year")["TotalBagged"].sum()

bagged_change = (
    (bagged_volume[2026] - bagged_volume[2025])
    / bagged_volume[2025] * 100
)

print("Bagged Volume:")
print(bagged_volume)

print(f"\nBagged volume change: {bagged_change:.2f}%")

Bagged Volume:
Year
2025    7.823154e+08
2026    9.197537e+08
Name: TotalBagged, dtype: float64

Bagged volume change: 17.57%


In [13]:
# Verify bagged share of total avocado volume
bagged_share = bagged_volume / total_volume * 100

bagged_share

Year
2025    41.819421
2026    43.266233
dtype: float64

In [14]:
# Verify growth by bag size
bag_columns = ["Sml Bagged", "Lrg Bagged", "X-Lrg Bagged"]

bag_sizes = comparison_data.groupby("Year")[bag_columns].sum()

bag_size_change = (
    (bag_sizes.loc[2026] - bag_sizes.loc[2025])
    / bag_sizes.loc[2025] * 100
)

bag_size_change

Sml Bagged       11.901338
Lrg Bagged       57.308364
X-Lrg Bagged    215.254259
dtype: float64

In [15]:
# Verify growth in avocado PLU categories
plu_columns = ["4046", "4225", "4770"]

plu_volume = comparison_data.groupby("Year")[plu_columns].sum()

plu_change = (
    (plu_volume.loc[2026] - plu_volume.loc[2025])
    / plu_volume.loc[2025] * 100
)

plu_change


4046      2.524413
4225     35.609882
4770    106.583906
dtype: float64